[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-og-helse/blob/main/uke05-multimodal-ai/01_bilde_tekst_clip_zero_shot_blomster.ipynb)


# 🌺 CLIP zero-shot på blomster

## Læringsmål
- Forstå bilde–tekst-felles representasjoner (CLIP)
- Utføre zero-shot klassifikasjon med tekstprompter
- Evaluere enkel nøyaktighet og inspisere topp-5 sannsynligheter


### 🔧 Miljøoppsett – fungerer både lokalt og i Google Colab


In [ ]:
import sys, subprocess, os, glob, random

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🚀 Kjører i Google Colab")
    # Lettere avhengigheter for Colab (bruk subprocess for å unngå notebook-magic avhengighet)
    try:
        import transformers  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "-q"])  # type: ignore
    try:
        import PIL  # noqa: F401
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow", "-q"])  # type: ignore

    if not os.path.exists('AI-og-helse'):
        print("📥 Laster ned kursmateriell...")
        subprocess.check_call(["git", "clone", "https://github.com/arvidl/AI-og-helse.git"])  # type: ignore
    if os.path.exists('AI-og-helse'):
        os.chdir('AI-og-helse')
        print(f"📁 Byttet til mappe: {os.getcwd()}")
else:
    print("💻 Kjører i lokal miljø")

import numpy as np
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
print("✅ Miljø klart")


In [ ]:
# Konfigurasjon
LABELS = ["daisy","dandelion","rose","sunflower","tulip"]
PROMPT_TEMPLATES = [
    "a photo of a {}",
    "a close-up photo of a {}",
    "a high quality image of a {}"
]

# Finn et lite utvalg av bilder per klasse for rask demo
BASE_DIR = os.path.join("data", "flowers")
SAMPLES_PER_CLASS = 5

def collect_image_paths(base_dir, labels, k=5):
    paths = []
    for label in labels:
        p = sorted(glob.glob(os.path.join(base_dir, label, "*.jpg")))
        if len(p) == 0:
            # fallback på png
            p = sorted(glob.glob(os.path.join(base_dir, label, "*.png")))
        random.shuffle(p)
        paths.extend([(pp, label) for pp in p[:k]])
    return paths

image_label_pairs = collect_image_paths(BASE_DIR, LABELS, SAMPLES_PER_CLASS)
print(f"Antall bilder valgt: {len(image_label_pairs)}")


In [ ]:
# Last modell og prosessor
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print(f"Modell lastet på {DEVICE}")


In [ ]:
# Hjelpefunksjoner

def build_prompts(labels, templates):
    prompts = []
    for l in labels:
        for t in templates:
            prompts.append(t.format(l))
    return prompts

ALL_PROMPTS = build_prompts(LABELS, PROMPT_TEMPLATES)

@torch.no_grad()
def clip_zero_shot(image_path, prompts):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(text=prompts, images=image, return_tensors="pt", padding=True).to(DEVICE)
    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=-1).squeeze(0).detach().cpu()
    return probs

# Kjør prediksjon på alle bilder og beregn enkel accuracy
results = []
for img_path, true_label in image_label_pairs:
    probs = clip_zero_shot(img_path, ALL_PROMPTS)
    # Aggreger over prompt-varianter per klasse: ta maks eller snitt; vi bruker snitt
    num_templates = len(PROMPT_TEMPLATES)
    class_scores = []
    for i, label in enumerate(LABELS):
        # indekser for denne klassens prompts
        start = i * num_templates
        end = (i + 1) * num_templates
        score = probs[start:end].mean().item()
        class_scores.append(score)
    class_scores = torch.tensor(class_scores)
    pred_idx = int(torch.argmax(class_scores))
    pred_label = LABELS[pred_idx]
    results.append((img_path, true_label, pred_label, class_scores.tolist()))

accuracy = sum(1 for _, t, p, _ in results if t == p) / max(1, len(results))
print(f"Enkel accuracy på utvalget: {accuracy:.2%}")


In [ ]:
# Visualisering av noen eksempler
import matplotlib.pyplot as plt

num_show = min(6, len(results))
fig, axes = plt.subplots(2, (num_show+1)//2, figsize=(12, 6))
axes = axes.flatten()
for ax, (img_path, true_label, pred_label, class_scores) in zip(axes, results[:num_show]):
    ax.imshow(Image.open(img_path))
    ax.set_title(f"GT: {true_label}\nPred: {pred_label}")
    ax.axis('off')
plt.tight_layout()
plt.show()

# Vis topp-5 for første bilde
if len(results) > 0:
    _, true_label, pred_label, class_scores = results[0]
    scores = torch.tensor(class_scores)
    topk = min(5, len(LABELS))
    vals, idxs = torch.topk(scores, topk)
    print("Top-5 klasser:")
    for v, i in zip(vals.tolist(), idxs.tolist()):
        print(f"  {LABELS[i]}: {v:.3f}")


### Refleksjon
- Hvorfor kan noen prompt-varianter fungere bedre enn andre?
- Når svikter zero-shot klassifikasjon på dette datasettet?
- Hvordan påvirker antall prompt-varianter resultatet?
